# Hybrid Retrieval: Combining BM25 and Semantic Search

BM25 excels at exact keyword matching but misses semantic relationships. Embedding-based search captures meaning but can miss important keywords. Hybrid retrieval combines both approaches to get the best of both worlds.

In this notebook, we'll implement a complete hybrid retrieval system using:
- BM25 for lexical matching (from our previous implementation)
- OpenAI embeddings for semantic search
- Score fusion to combine results

> **Where are we in the course?** In the previous notebook, we implemented BM25 for keyword-based search. Now we'll combine it with the semantic search from our earlier embeddings work.

In [ ]:
%pip install -q openai numpy

In [ ]:
import os
import math
import numpy as np
from getpass import getpass
from collections import Counter
from dataclasses import dataclass
import re
from openai import OpenAI

# API Key Setup
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI()
EMBED_MODEL = "text-embedding-3-small"

## Part 1: BM25 Implementation (Recap)

First, let's bring in our BM25 implementation from the previous notebook.

In [ ]:
def tokenize(text: str) -> list[str]:
    """Simple tokenizer."""
    text = text.lower()
    tokens = re.findall(r'\b\w+\b', text)
    return [t for t in tokens if len(t) > 1]

@dataclass
class BM25:
    """BM25 ranking implementation."""
    k1: float = 1.5
    b: float = 0.75
    
    def __post_init__(self):
        self.corpus = []
        self.doc_freqs = []
        self.doc_lengths = []
        self.avgdl = 0
        self.idf = {}
        self.n_docs = 0
    
    def index(self, documents: list[str]):
        self.corpus = documents
        self.n_docs = len(documents)
        self.doc_freqs = []
        self.doc_lengths = []
        doc_containing_term = Counter()
        
        for doc in documents:
            tokens = tokenize(doc)
            self.doc_lengths.append(len(tokens))
            freq = Counter(tokens)
            self.doc_freqs.append(freq)
            for term in set(tokens):
                doc_containing_term[term] += 1
        
        self.avgdl = sum(self.doc_lengths) / self.n_docs if self.n_docs > 0 else 0
        self.idf = {}
        for term, n in doc_containing_term.items():
            self.idf[term] = math.log((self.n_docs - n + 0.5) / (n + 0.5) + 1)
    
    def score(self, query: str, doc_idx: int) -> float:
        query_terms = tokenize(query)
        doc_freq = self.doc_freqs[doc_idx]
        doc_len = self.doc_lengths[doc_idx]
        
        score = 0.0
        for term in query_terms:
            if term not in self.idf:
                continue
            tf = doc_freq.get(term, 0)
            idf = self.idf[term]
            numerator = tf * (self.k1 + 1)
            denominator = tf + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)
            score += idf * (numerator / denominator)
        return score
    
    def search(self, query: str, top_k: int = 10) -> list[tuple[int, float]]:
        scores = [(i, self.score(query, i)) for i in range(self.n_docs)]
        scores.sort(key=lambda x: x[1], reverse=True)
        return [(idx, score) for idx, score in scores[:top_k] if score > 0]

## Part 2: Semantic Search with Embeddings

Now let's implement semantic search using OpenAI embeddings and cosine similarity.

In [ ]:
def get_embedding(text: str) -> list[float]:
    """Get embedding for a single text."""
    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=text
    )
    return response.data[0].embedding

def get_embeddings_batch(texts: list[str]) -> list[list[float]]:
    """Get embeddings for multiple texts in one API call."""
    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=texts
    )
    return [item.embedding for item in response.data]

def cosine_similarity(a: list[float], b: list[float]) -> float:
    """Compute cosine similarity between two vectors."""
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [ ]:
@dataclass
class VectorSearch:
    """Simple vector search using OpenAI embeddings."""
    
    def __post_init__(self):
        self.corpus = []
        self.embeddings = []
    
    def index(self, documents: list[str]):
        """Index documents by computing their embeddings."""
        self.corpus = documents
        print(f"Computing embeddings for {len(documents)} documents...")
        self.embeddings = get_embeddings_batch(documents)
        print(f"Indexed {len(self.embeddings)} documents")
    
    def search(self, query: str, top_k: int = 10) -> list[tuple[int, float]]:
        """Search for similar documents."""
        query_embedding = get_embedding(query)
        
        # Compute similarities
        scores = []
        for idx, doc_embedding in enumerate(self.embeddings):
            sim = cosine_similarity(query_embedding, doc_embedding)
            scores.append((idx, sim))
        
        # Sort by similarity (descending)
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]

## Part 3: Test Corpus

Let's create a test corpus and compare BM25 vs Vector search.

In [ ]:
documents = [
    "Python is a versatile programming language used for web development, data science, and automation.",
    "Machine learning algorithms learn patterns from data to make predictions without explicit programming.",
    "Deep neural networks consist of multiple layers that transform input data into useful representations.",
    "Natural language processing enables computers to understand, interpret, and generate human language.",
    "SQL databases store structured data in tables with rows and columns, supporting complex queries.",
    "NoSQL databases like MongoDB store unstructured data in flexible document formats.",
    "RESTful APIs use HTTP methods to enable communication between software systems.",
    "Docker containers package applications with their dependencies for consistent deployment.",
    "Kubernetes orchestrates containerized applications across clusters of machines.",
    "Git version control tracks changes in source code and enables team collaboration.",
    "Agile methodology emphasizes iterative development and continuous feedback.",
    "Test-driven development writes tests before implementing the actual code.",
]

# Index with both methods
bm25 = BM25()
bm25.index(documents)

vector_search = VectorSearch()
vector_search.index(documents)

In [ ]:
def compare_searches(query: str, top_k: int = 5):
    """Compare BM25 and vector search results."""
    print(f"Query: '{query}'")
    print("=" * 70)
    
    bm25_results = bm25.search(query, top_k)
    vector_results = vector_search.search(query, top_k)
    
    print(f"\n{'BM25 Results':<35} | {'Vector Search Results':<35}")
    print("-" * 70)
    
    for i in range(max(len(bm25_results), len(vector_results))):
        bm25_str = ""
        vector_str = ""
        
        if i < len(bm25_results):
            idx, score = bm25_results[i]
            bm25_str = f"[{score:.2f}] Doc {idx}"
        
        if i < len(vector_results):
            idx, score = vector_results[i]
            vector_str = f"[{score:.3f}] Doc {idx}"
        
        print(f"{bm25_str:<35} | {vector_str:<35}")
    print()

# Test with different query types
test_queries = [
    "Python programming",           # Exact keyword match
    "how to store data",             # Semantic - should find database docs
    "coding best practices",         # Semantic - should find TDD, Agile
    "ML algorithms",                 # Abbreviation + keyword
]

for query in test_queries:
    compare_searches(query)

## Part 4: Hybrid Retrieval

Now let's combine BM25 and vector search. The challenge is that their scores are on different scales:
- BM25: typically 0-20+
- Cosine similarity: 0-1

We need to normalize scores before combining them.

In [ ]:
def normalize_scores(results: list[tuple[int, float]]) -> list[tuple[int, float]]:
    """Normalize scores to 0-1 range using min-max normalization."""
    if not results:
        return []
    
    scores = [score for _, score in results]
    min_score = min(scores)
    max_score = max(scores)
    
    if max_score == min_score:
        return [(idx, 1.0) for idx, _ in results]
    
    return [
        (idx, (score - min_score) / (max_score - min_score))
        for idx, score in results
    ]

In [ ]:
@dataclass
class HybridSearch:
    """Hybrid search combining BM25 and vector search."""
    alpha: float = 0.5  # Weight for BM25 (1-alpha for vector)
    
    def __post_init__(self):
        self.bm25 = BM25()
        self.vector = VectorSearch()
        self.corpus = []
    
    def index(self, documents: list[str]):
        """Index documents with both methods."""
        self.corpus = documents
        self.bm25.index(documents)
        self.vector.index(documents)
    
    def search(self, query: str, top_k: int = 10) -> list[tuple[int, float, dict]]:
        """Hybrid search with score breakdown."""
        # Get results from both methods
        bm25_results = self.bm25.search(query, top_k=len(self.corpus))
        vector_results = self.vector.search(query, top_k=len(self.corpus))
        
        # Normalize scores
        bm25_normalized = normalize_scores(bm25_results)
        # Vector results are already 0-1 but normalize anyway for consistency
        vector_normalized = normalize_scores(vector_results)
        
        # Create score dictionaries
        bm25_scores = {idx: score for idx, score in bm25_normalized}
        vector_scores = {idx: score for idx, score in vector_normalized}
        
        # Combine scores
        combined = []
        all_docs = set(bm25_scores.keys()) | set(vector_scores.keys())
        
        for idx in all_docs:
            bm25_score = bm25_scores.get(idx, 0)
            vector_score = vector_scores.get(idx, 0)
            
            # Weighted combination
            hybrid_score = self.alpha * bm25_score + (1 - self.alpha) * vector_score
            
            combined.append((idx, hybrid_score, {
                "bm25": bm25_score,
                "vector": vector_score,
                "hybrid": hybrid_score
            }))
        
        # Sort by hybrid score
        combined.sort(key=lambda x: x[1], reverse=True)
        return combined[:top_k]

# Test hybrid search
hybrid = HybridSearch(alpha=0.5)
hybrid.index(documents)

In [ ]:
def display_hybrid_results(query: str, top_k: int = 5):
    """Display hybrid search results with score breakdown."""
    print(f"Query: '{query}'")
    print("=" * 80)
    
    results = hybrid.search(query, top_k)
    
    print(f"{'Rank':<5} {'Doc':<5} {'Hybrid':<10} {'BM25':<10} {'Vector':<10} {'Preview':<40}")
    print("-" * 80)
    
    for rank, (idx, score, breakdown) in enumerate(results, 1):
        preview = documents[idx][:37] + "..."
        print(f"{rank:<5} {idx:<5} {breakdown['hybrid']:.3f}     {breakdown['bm25']:.3f}     {breakdown['vector']:.3f}     {preview}")
    print()

# Test queries
display_hybrid_results("Python programming language")
display_hybrid_results("how to store and query data")
display_hybrid_results("deployment and containers")

## Part 5: Tuning the Alpha Parameter

The `alpha` parameter controls the balance between BM25 and vector search:
- `alpha=1.0`: Pure BM25
- `alpha=0.5`: Equal weight
- `alpha=0.0`: Pure vector search

The optimal value depends on your use case.

In [ ]:
def compare_alphas(query: str, alphas: list[float] = [0.0, 0.3, 0.5, 0.7, 1.0]):
    """Compare results at different alpha values."""
    print(f"Query: '{query}'")
    print("=" * 80)
    
    for alpha in alphas:
        hybrid.alpha = alpha
        results = hybrid.search(query, top_k=3)
        
        label = "Vector" if alpha == 0 else "BM25" if alpha == 1 else f"α={alpha}"
        doc_indices = [str(idx) for idx, _, _ in results]
        print(f"{label:<10}: Top docs = [{', '.join(doc_indices)}]")
    
    print()

compare_alphas("Python programming")
compare_alphas("how to store information")  # Semantic query
compare_alphas("SQL database queries")      # Keyword-heavy query

## Part 6: Score Interpretation

Understanding when each method contributes to the final ranking:

In [ ]:
def analyze_score_contributions(query: str):
    """Analyze how BM25 and vector search contribute to rankings."""
    hybrid.alpha = 0.5
    results = hybrid.search(query, top_k=5)
    
    print(f"Query: '{query}'")
    print("=" * 60)
    
    for rank, (idx, score, breakdown) in enumerate(results, 1):
        bm25_contrib = breakdown['bm25'] * 0.5
        vector_contrib = breakdown['vector'] * 0.5
        
        # Determine which method contributed more
        if bm25_contrib > vector_contrib * 1.5:
            dominant = "BM25 dominant"
        elif vector_contrib > bm25_contrib * 1.5:
            dominant = "Vector dominant"
        else:
            dominant = "Balanced"
        
        print(f"\nRank {rank}: Doc {idx} [{dominant}]")
        print(f"  BM25 contribution:   {bm25_contrib:.3f} ({bm25_contrib/score*100:.0f}%)")
        print(f"  Vector contribution: {vector_contrib:.3f} ({vector_contrib/score*100:.0f}%)")
        print(f"  Document: {documents[idx][:60]}...")

analyze_score_contributions("machine learning algorithms")
print()
analyze_score_contributions("coding methodology practices")

---

## Exercises

### Exercise 1: Dynamic Alpha Selection

Implement a function that automatically selects alpha based on query characteristics. For example:
- Queries with rare/specific terms → higher BM25 weight
- Queries with common/vague terms → higher vector weight

In [ ]:
def dynamic_alpha(query: str, bm25_index: BM25) -> float:
    """Compute optimal alpha based on query characteristics."""
    # Hint: Consider:
    # - Average IDF of query terms (high IDF = rare terms = trust BM25 more)
    # - Query length (longer queries might benefit from semantic)
    # - Presence of stopwords
    pass

### Exercise 2: Late Fusion vs Early Fusion

Our current implementation uses "late fusion" (combine after retrieving from both). Implement "early fusion" where you:
1. Concatenate BM25 features with embeddings
2. Learn a combined similarity function

In [ ]:
# Your implementation here

### Exercise 3: Multi-Vector Hybrid

Extend hybrid search to support multiple vector representations:
- Title embedding
- Body embedding
- Summary embedding

Each with its own weight in the final score.

In [ ]:
# Your implementation here

---

## Summary

- Hybrid retrieval combines lexical (BM25) and semantic (vector) search
- Score normalization is essential when combining different ranking functions
- The alpha parameter controls the balance between methods
- Different query types benefit from different alpha values
- Production systems often use adaptive alpha selection

In the next notebook, we'll explore query expansion with LLMs - using language models to generate alternative phrasings of search queries to improve recall.